In [ ]:
# -*- coding: utf-8 -*-
"""
Slope-unit model training + comparison + EDA + ensemble + model saving
No spatial grouping
"""

# ==========================================
# 1. INSTALL & IMPORT LIBRARIES
# ==========================================
!pip install openpyxl scikit-optimize shap xgboost joblib -q

import os
import gc
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from google.colab import drive

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold, cross_val_predict,
    cross_val_score
)
from sklearn.base import clone
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, auc,
    precision_score, recall_score, f1_score
)
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

In [6]:
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

import shutil # Import shutil for directory operations

mountpoint = '/content/drive'

# Attempt to unmount first if already mounted
if os.path.exists(mountpoint) and os.path.ismount(mountpoint):
    print("Drive is currently mounted. Attempting to unmount.")
    try:
        drive.flush_and_unmount()
        print("Drive unmounted successfully.")
    except Exception as e:
        print(f"Error unmounting drive: {e}. Will try to clear the directory.")

# Ensure the mountpoint directory is empty or doesn't exist
if os.path.isdir(mountpoint) and os.listdir(mountpoint):
    print(f"Mountpoint {mountpoint} contains files after unmount attempt. Removing contents.")
    try:
        shutil.rmtree(mountpoint) # Remove the directory and its contents
        os.makedirs(mountpoint)    # Recreate an empty directory
        print(f"Contents of {mountpoint} removed and directory recreated.")
    except Exception as e:
        print(f"Error clearing/recreating mountpoint directory: {e}. Trying to mount directly.")
elif not os.path.isdir(mountpoint):
    # If the directory doesn't exist, create it
    os.makedirs(mountpoint, exist_ok=True)

drive.mount(mountpoint, force_remount=True)

Drive is currently mounted. Attempting to unmount.
Drive unmounted successfully.
Mounted at /content/drive


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

mpl.rcParams.update({
    "font.family":         "DejaVu Sans",
    "font.size":           11,
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      11,
    "axes.linewidth":      1.0,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
    "legend.fontsize":     10,
    "legend.frameon":      True,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.5",
    "figure.dpi":          120,
    "savefig.dpi":         600,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

MODEL_COLOURS = {
    "RF":       "#2E86AB",
    "SVM":      "#E07A5F",
    "XGB":      "#7E4E90",
    "Ensemble": "#0E1B2C",
}

PRETTY_FEATURE = {
    "aspect":         "Aspect",
    "curvature":      "Curvature (profile)",
    "dis_to_fault":   "Distance to fault",
    "dis_to_river":   "Distance to river",
    "dis_to_road":    "Distance to road",
    "dis_to_thrust":  "Distance to thrust",
    "ndvi":           "NDVI",
    "slope":          "Slope",
    "twi":            "TWI",
    "aspect_average":        "Aspect (mean)",
    "aspect_stddev":         "Aspect (sd)",
    "curvature_average":     "Curvature (mean)",
    "curvature_stddev":      "Curvature (sd)",
    "dis_to_fault_average":  "Dist. to fault (mean)",
    "dis_to_fault_stddev":   "Dist. to fault (sd)",
    "dis_to_river_average":  "Dist. to river (mean)",
    "dis_to_river_stddev":   "Dist. to river (sd)",
    "dis_to_road_average":   "Dist. to road (mean)",
    "dis_to_road_stddev":    "Dist. to road (sd)",
    "dis_to_thrust_average": "Dist. to thrust (mean)",
    "dis_to_thrust_stddev":  "Dist. to thrust (sd)",
    "ndvi_average":          "NDVI (mean)",
    "ndvi_stddev":           "NDVI (sd)",
    "slope_average":         "Slope (mean)",
    "slope_stddev":          "Slope (sd)",
    "twi_average":           "TWI (mean)",
    "twi_stddev":            "TWI (sd)",
}

def pretty(name):
    return PRETTY_FEATURE.get(name, name)


In [ ]:
# ==========================================
# 2. PATHS
# ==========================================
data_path = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/slopeunits/"
grid_files = {
    "3x3": "ls_slp_3x3.csv",
    "15x15": "ls_slp_15x15.csv",
    "25x25": "ls_slp_25x25.csv"
}

base_output_dir = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/slope_unit_output"
os.makedirs(base_output_dir, exist_ok=True)

# ==========================================
# 3. SETTINGS
# ==========================================
# Drop these if present
DROP_COL = ["ndwi_average", "ndwi_stddev", "roughness_average", "roughness_stddev"]
TARGET_COL = "ID"
ID_COL = "fid"

RANDOM_STATE = 42
TEST_SIZE = 0.2
INNER_CV_SPLITS = 5
BAYES_N_ITER = 20

# Spatial-CV block size in metres (UTM 44N, EPSG:32644).
# Slope-unit polygon mean radius is roughly 0.3 km; a 2 km block keeps
# neighbouring units in the same fold, the slope-unit counterpart to the
# 1000 m block size used in the pixel pipeline.
SLOPE_UNIT_BLOCK_M = 2000

SHAP_BACKGROUND_SIZE = 100
SHAP_EXPLAIN_SIZE = 100

# ==========================================
# 4. HELPERS
# ==========================================
def make_dir(path):
    os.makedirs(path, exist_ok=True)

def load_table(path):
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    df.columns = df.columns.str.strip()
    return df

def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=4)

def plot_correlation_matrix(df, save_path, title):
    corr = df.rename(columns={c: pretty(c) for c in df.columns}).corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10, 8.5))   # slope-unit has more features -> wider
    sns.heatmap(
        corr, ax=ax,
        cmap="RdBu_r", center=0, vmin=-1, vmax=1,
        annot=True, fmt=".2f", annot_kws={"fontsize": 7},
        square=True, linewidths=0.4, linecolor="white",
        cbar_kws={"shrink": 0.75, "label": "Pearson r"},
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)

def plot_distributions_by_class(df, features, target_col, out_dir, grid_name):
    """Single 3xN composite KDE figure (slope-unit dataset can have up to 18 features)."""
    df_plot = df.copy()
    df_plot["class"] = df_plot[target_col].map({0: "Non-landslide", 1: "Landslide"})

    feats = list(features)
    n = len(feats)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.0 * nrows))
    axes = np.atleast_2d(axes).flatten()

    for ax, col in zip(axes, feats):
        sns.kdeplot(
            data=df_plot, x=col, hue="class",
            fill=True, common_norm=False, alpha=0.45, linewidth=1.6,
            palette={"Landslide": "#C0392B", "Non-landslide": "#2E86AB"},
            ax=ax, legend=False,
        )
        ax.set_title(pretty(col), fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel("Density")
        ax.grid(alpha=0.25, linestyle=":")

    for ax in axes[n:]:
        ax.axis("off")

    handles = [
        mpatches.Patch(facecolor="#C0392B", alpha=0.5, label="Landslide"),
        mpatches.Patch(facecolor="#2E86AB", alpha=0.5, label="Non-landslide"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"Conditioning-factor distributions by class - {grid_name}", y=1.00)
    fig.tight_layout(rect=(0, 0.03, 1, 0.97))

    save_path = os.path.join(out_dir, f"{grid_name}_distributions.png")
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": float(roc_auc_score(y_true, y_prob))
    }

def save_confusion_matrix(cm, labels, title, save_path):
    cm = np.asarray(cm)
    cm_pct = cm.astype(float) / cm.sum() * 100.0
    annot = np.array([[f"{cm[i, j]:d}\n({cm_pct[i, j]:.1f}%)"
                       for j in range(cm.shape[1])]
                      for i in range(cm.shape[0])])
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    sns.heatmap(
        cm, ax=ax, annot=annot, fmt="", cmap="Blues",
        xticklabels=labels, yticklabels=labels,
        cbar=False, annot_kws={"fontsize": 10},
        linewidths=0.4, linecolor="white",
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)

def save_roc_curve(y_true, y_prob, title, save_path, color="#2E86AB"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5.0, 4.2))
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"AUC = {auc_score:.3f}")
    ax.plot([0, 1], [0, 1], color="0.4", linestyle="--", linewidth=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.grid(alpha=0.25, linestyle=":")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)


def save_combined_roc(probs_dict, y_test, grid_name, save_path):
    """Overlay ROC curves for all models on one panel."""
    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    ax.plot([0, 1], [0, 1], color="0.4", linestyle="--", linewidth=1)
    for name, prob in probs_dict.items():
        fpr, tpr, _ = roc_curve(y_test, prob)
        ax.plot(fpr, tpr, linewidth=1.8,
                color=MODEL_COLOURS.get(name, None),
                label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC curves - {grid_name}")
    ax.grid(alpha=0.25, linestyle=":")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)

def plot_feature_importance(importances, feature_names, title, save_path, top_n=None, color="#2E86AB"):
    imp_df = (pd.DataFrame({"feature": [pretty(f) for f in feature_names],
                            "importance": importances})
                .sort_values("importance", ascending=False).head(top_n))
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(imp_df) + 1.5))
    sns.barplot(data=imp_df, x="importance", y="feature",
                color=color, edgecolor="white", linewidth=0.5, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Mean decrease in Gini / gain (normalised)")
    ax.set_ylabel("Conditioning factor")
    ax.grid(axis="x", alpha=0.25, linestyle=":")
    fig.tight_layout()
    fig.savefig(save_path)
    plt.show()
    plt.close(fig)

    return imp_df

def get_native_importance(model_name, fitted_estimator, X_data, y_data):
    if model_name in ["RF", "XGB"]:
        return fitted_estimator.feature_importances_
    elif model_name == "SVM":
        perm = permutation_importance(
            fitted_estimator,
            X_data,
            y_data,
            scoring="f1",
            n_repeats=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        return perm.importances_mean
    else:
        raise ValueError("Unknown model name")

def ensemble_predict_proba(model_dict, X):
    probs = []
    for _, model in model_dict.items():
        probs.append(model.predict_proba(X)[:, 1])
    return np.mean(np.column_stack(probs), axis=1)

def plot_shap_tree(model, X_sample, title, save_prefix):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # Normalise SHAP output to a 2-D (n_samples, n_features) array for the
    # positive (landslide) class. shap/sklearn versions differ in what
    # TreeExplainer returns:
    #   * list [class0, class1] of 2-D arrays           (older shap)
    #   * 3-D ndarray (n_samples, n_features, n_classes) (newer shap on RF)
    #   * plain 2-D ndarray                              (single-output, e.g. XGB)
    # Without this, RandomForest's 3-D output reached summary_plot and was
    # drawn as a clipped interaction-style matrix instead of a beeswarm.
    shap_plot_values = shap_values
    if isinstance(shap_plot_values, list):
        shap_plot_values = (shap_plot_values[1]
                            if len(shap_plot_values) > 1 else shap_plot_values[0])
    shap_plot_values = np.asarray(shap_plot_values)
    if shap_plot_values.ndim == 3:
        cls = 1 if shap_plot_values.shape[-1] > 1 else 0
        shap_plot_values = shap_plot_values[..., cls]

    plt.figure(figsize=(7, 5))
    shap.summary_plot(shap_plot_values, X_sample, show=False,
                      max_display=X_sample.shape[1],
                      feature_names=[pretty(f) for f in X_sample.columns])
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_shap_summary.png", dpi=600, bbox_inches="tight")
    plt.show()
    plt.close()

def plot_shap_kernel(model, X_background, X_sample, title, save_prefix):
    X_bg = shap.sample(
        X_background,
        min(len(X_background), SHAP_BACKGROUND_SIZE),
        random_state=RANDOM_STATE
    )
    X_explain = shap.sample(
        X_sample,
        min(len(X_sample), SHAP_EXPLAIN_SIZE),
        random_state=RANDOM_STATE
    )

    predict_fn = lambda data: model.predict_proba(
        pd.DataFrame(data, columns=X_sample.columns)
    )[:, 1]

    explainer = shap.KernelExplainer(predict_fn, X_bg)
    shap_values = explainer.shap_values(X_explain, nsamples=100)

    plt.figure()
    shap.summary_plot(shap_values, X_explain, show=False,
                      max_display=X_explain.shape[1],
                      feature_names=[pretty(f) for f in X_explain.columns])
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_shap_summary.png", dpi=600, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_ensemble_shap(models_dict, X_train, X_test, save_prefix, title):
    """Kernel-SHAP summary for the equal-weight ensemble."""
    X_bg = shap.sample(X_train, min(len(X_train), SHAP_BACKGROUND_SIZE), random_state=RANDOM_STATE)
    X_explain = shap.sample(X_test, min(len(X_test), SHAP_EXPLAIN_SIZE), random_state=RANDOM_STATE)

    predict_fn = lambda data: ensemble_predict_proba(
        models_dict, pd.DataFrame(data, columns=X_test.columns)
    )

    explainer = shap.KernelExplainer(predict_fn, X_bg)
    shap_values = explainer.shap_values(X_explain, nsamples=100)

    plt.figure(figsize=(7, 5))
    shap.summary_plot(shap_values, X_explain, show=False,
                      max_display=X_explain.shape[1],
                      feature_names=[pretty(f) for f in X_explain.columns])
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_shap_summary.png", dpi=600, bbox_inches="tight")
    plt.show()
    plt.close()
def find_best_threshold(y_true, y_prob, method="f1"):
    """
    Find the best classification threshold based on recall, f1, or balanced score.

    Parameters
    ----------
    y_true : array-like
        True binary labels.
    y_prob : array-like
        Predicted probabilities for positive class.
    method : str
        One of: "recall", "f1", "balanced"

    Returns
    -------
    best_threshold : float
    best_score : float
    best_metrics : dict
    """
    thresholds = np.linspace(0.01, 0.99, 99)

    best_threshold = 0.5
    best_score = -1
    best_metrics = None

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if method == "recall":
            score = recall
        elif method == "f1":
            score = f1
        elif method == "balanced":
            score = (precision + recall) / 2
        else:
            raise ValueError("method must be one of: 'recall', 'f1', 'balanced'")

        if score > best_score:
            best_score = score
            best_threshold = t
            best_metrics = {
                "precision": precision,
                "recall": recall,
                "f1": f1
            }

    return best_threshold, best_score, best_metrics

def plot_ensemble_permutation_importance(models_dict, X_test, y_test, save_path, title, threshold=0.5):
    baseline_prob = ensemble_predict_proba(models_dict, X_test)
    baseline_pred = (baseline_prob >= threshold).astype(int)
    baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)

    importances = []
    rng = np.random.default_rng(RANDOM_STATE)

    for col in X_test.columns:
        scores = []
        for _ in range(10):
            X_perm = X_test.copy()
            X_perm[col] = rng.permutation(X_perm[col].values)
            prob = ensemble_predict_proba(models_dict, X_perm)
            pred = (prob >= threshold).astype(int)
            score = f1_score(y_test, pred, zero_division=0)
            scores.append(baseline_f1 - score)
        importances.append(np.mean(scores))

    return plot_feature_importance(
        np.array(importances),
        list(X_test.columns),
        title=title,
        save_path=save_path
    )


def autodrop_correlated(X, threshold=0.8, label="", prefer_drop=("roughness", "ndwi")):
    """Iteratively drop features where |Pearson r| > threshold.
    Returns
    -------
    X_clean : DataFrame
    dropped_list : list of (dropped_col, retained_col, r_value) tuples
    """
    cols = list(X.columns)
    dropped = []

    while True:
        corr = X[cols].corr(numeric_only=True).abs()
        found = False
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                r = corr.iloc[i, j]
                if r > threshold:
                    di = any(tok in cols[i] for tok in prefer_drop)
                    dj = any(tok in cols[j] for tok in prefer_drop)
                    if di != dj:
                        # prefer dropping flagged collinear proxy (roughness, ndwi),
                        # i.e. keep the conventional factor (slope, ndvi)
                        to_drop, to_keep = (cols[i], cols[j]) if di else (cols[j], cols[i])
                    else:
                        ci = int((corr[cols[i]] > threshold).sum()) - 1
                        cj = int((corr[cols[j]] > threshold).sum()) - 1
                        if ci > cj or (ci == cj and cols[i] > cols[j]):
                            to_drop, to_keep = cols[i], cols[j]
                        else:
                            to_drop, to_keep = cols[j], cols[i]
                    dropped.append((to_drop, to_keep, round(float(r), 4)))
                    cols.remove(to_drop)
                    found = True
                    break
            if found:
                break
        if not found:
            break

    tag = f" [{label}]" if label else ""
    if dropped:
        print(f"[autodrop{tag}] Removed {len(dropped)} feature(s) with |r| > {threshold}:")
        for d, k, rv in dropped:
            print(f"  DROP '{d}'  (|r|={rv} with '{k}')")
    else:
        print(f"[autodrop{tag}] No features exceed |r| > {threshold}.")
    return X[cols].copy(), dropped

# ==========================================
# 5. MODELS + SEARCH SPACES
# ==========================================
models = {
    "RF": RandomForestClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "XGB": XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )
}

param_spaces = {
    "RF": {
        "n_estimators": Integer(150, 400),
        "max_depth": Integer(5, 25),
        "min_samples_split": Integer(2, 10),
        "min_samples_leaf": Integer(1, 10),
        "class_weight": Categorical(["balanced", "balanced_subsample"])
    },
    "SVM": {
        "model__C": Real(1e-2, 100, prior="log-uniform"),
        "model__gamma": Real(1e-5, 10, prior="log-uniform"),
        "model__kernel": Categorical(["rbf"]),
        "model__class_weight": Categorical(["balanced"])
    },
    "XGB": {
        "n_estimators": Integer(150, 400),
        "max_depth": Integer(3, 12),
        "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
        "subsample": Real(0.6, 1.0),
        "colsample_bytree": Real(0.6, 1.0),
    }
}

# ==========================================
# 6. LOOP OVER ALL GRIDS
# ==========================================
all_grid_summary = []

def compute_spatial_cv_auc_stats(estimator, X, y, groups, n_splits=5, random_state=RANDOM_STATE):
    """Spatially-blocked cross-validated ROC-AUC (mean, std, fold scores).

    Identical protocol to the grid/pixel model-comparison notebook so both mapping-unit
    systems report comparable spatial_cv_auc_* values: StratifiedGroupKFold over the spatial
    blocks, scoring='roc_auc', evaluated on the TRAINING fold only (AUC is threshold-free).
    """
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    auc_scores = cross_val_score(estimator, X, y, cv=cv, groups=groups,
                                 scoring="roc_auc", n_jobs=-1)
    return float(np.mean(auc_scores)), float(np.std(auc_scores)), auc_scores


for grid, file in grid_files.items():
    print(f"\n{'='*25} GRID: {grid} {'='*25}\n")

    grid_out_dir = os.path.join(base_output_dir, grid)
    make_dir(grid_out_dir)
    make_dir(os.path.join(grid_out_dir, "models"))
    make_dir(os.path.join(grid_out_dir, "plots"))
    make_dir(os.path.join(grid_out_dir, "plots", "roc"))
    make_dir(os.path.join(grid_out_dir, "plots", "cm"))
    make_dir(os.path.join(grid_out_dir, "plots", "importance"))
    make_dir(os.path.join(grid_out_dir, "plots", "shap"))

    # --------------------------------------
    # Load data
    # --------------------------------------
    df = load_table(os.path.join(data_path, file))

    # drop rows with any null values
    df = df.dropna().reset_index(drop=True)

    # ---- attach slope-unit centroid coords for spatial cross-validation ----
    gpkg_path = os.path.join(data_path, f"su_{grid}.gpkg")
    gdf_units = gpd.read_file(gpkg_path)
    if gdf_units.crs is None or gdf_units.crs.to_epsg() != 32644:
        gdf_units = gdf_units.to_crs(epsg=32644)
    gdf_units = gdf_units.reset_index(drop=True)
    gdf_units["fid"] = gdf_units.index + 1
    _cent = gdf_units.geometry.centroid
    _cent_df = pd.DataFrame({
        "fid": gdf_units["fid"].values,
        "_cx_m": _cent.x.values,
        "_cy_m": _cent.y.values,
    })
    df = df.merge(_cent_df, on="fid", how="left")
    _n_no_xy = int(df[["_cx_m", "_cy_m"]].isna().any(axis=1).sum())
    if _n_no_xy:
        print(f"  WARNING: dropping {_n_no_xy} rows without centroid coords")
    df = df.dropna(subset=["_cx_m", "_cy_m"]).reset_index(drop=True)
    print(f"  joined centroids; rows with X,Y: {len(df)}")

    if TARGET_COL not in df.columns:
        raise ValueError(f"'{TARGET_COL}' not found in {file}")

    # keep only target + predictors; do not use fid as feature
    # DROP_COL removes known excluded columns (ndwi/roughness variants)
    meta_cols = [c for c in [TARGET_COL, ID_COL, "_cx_m", "_cy_m"] + DROP_COL if c in df.columns]
    X_all = df.drop(columns=meta_cols, errors="ignore").copy()
    y = df[TARGET_COL].astype(int).values

    print("Shape after dropna:", df.shape)
    print("Class balance:", np.bincount(y))
    print("Features (before auto-drop):", list(X_all.columns))

    # -- Gross correlation heatmap (all retained features before auto-drop)
    plot_correlation_matrix(
        X_all,
        save_path=os.path.join(grid_out_dir, "plots", f"{grid}_gross_correlation_matrix.png"),
        title=f"All-features Correlation Matrix (pre-drop) - {grid}"
    )

    # -- Auto-drop features with |r| > 0.8 --------------------------------
    # DROP_COL already excluded ndwi/roughness variants above.
    # This catches any remaining correlated pairs in the mean/stddev features.
    X, dropped_features = autodrop_correlated(X_all, threshold=0.8, label=grid)
    print("Features (after auto-drop):", list(X.columns))

    # -- Post-drop correlation heatmap ------------------------------------
    plot_correlation_matrix(
        X,
        save_path=os.path.join(grid_out_dir, "plots", f"{grid}_correlation_matrix.png"),
        title=f"Retained-features Correlation Matrix - {grid}"
    )

    plot_distributions_by_class(
        df=pd.concat([X, pd.Series(y, name=TARGET_COL)], axis=1),
        features=X.columns,
        target_col=TARGET_COL,
        out_dir=os.path.join(grid_out_dir, "plots"),
        grid_name=grid
    )

    # --------------------------------------
    # SPATIAL + STRATIFIED train-test split
    # --------------------------------------
    _bx = np.floor(df["_cx_m"].values / SLOPE_UNIT_BLOCK_M).astype(int)
    _by = np.floor(df["_cy_m"].values / SLOPE_UNIT_BLOCK_M).astype(int)
    groups = pd.Series(
        pd.Series(_bx).astype(str) + "_" + pd.Series(_by).astype(str),
        index=range(len(df)),
    )
    print(f"  spatial groups (block={SLOPE_UNIT_BLOCK_M} m): {groups.nunique()}")

    outer_splitter = StratifiedGroupKFold(
        n_splits=int(round(1.0 / TEST_SIZE)),   # TEST_SIZE=0.2 -> 5 folds
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    train_idx, test_idx = next(outer_splitter.split(X, y, groups=groups))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_train = y[train_idx]
    y_test  = y[test_idx]
    groups_train = groups.iloc[train_idx].reset_index(drop=True)
    groups_test  = groups.iloc[test_idx].reset_index(drop=True)
    print(f"  train n={len(X_train)} pos={int(y_train.sum())} | "
          f"test n={len(X_test)} pos={int(y_test.sum())}")
    print(f"  train groups={groups_train.nunique()}, test groups={groups_test.nunique()}")

    inner_cv = StratifiedGroupKFold(
        n_splits=INNER_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    # XGB receives scale_pos_weight from the training-fold imbalance
    _n_neg = int((y_train == 0).sum())
    _n_pos = int((y_train == 1).sum())
    scale_pos_weight = _n_neg / max(1, _n_pos)
    models["XGB"] = XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
    )
    print(f"  XGB scale_pos_weight for this grid: {scale_pos_weight:.3f}")
    oof_probs = {}
    model_thresholds = {}

    fitted_models = {}
    fitted_probs = {}      # Collect each model's test-set probabilities for the combined ROC panel
    grid_results = []

    # ======================================
    # 7. TRAIN RF, SVM, XGB
    # ======================================
    for name, model in models.items():
        print(f"\n--------------- Training {name} on {grid} ---------------")

        if name == "SVM":
            estimator = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])
            search_space = param_spaces[name]
        else:
            estimator = model
            search_space = param_spaces[name]

        search = BayesSearchCV(
            estimator=estimator,
            search_spaces=search_space,
            n_iter=BAYES_N_ITER,
            cv=inner_cv,
            scoring="f1",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True
        )

        search.fit(X_train, y_train, groups=groups_train)

        # Tune the F1-optimal threshold from OOF predictions on the TRAINING
        # set only (removes the previous test-set selection bias).  Both the
        # outer (inner_cv) and inner Platt folds are spatially blocked
        # (audit fix M1): the outer is StratifiedGroupKFold from above;
        # the inner Platt cv is a precomputed 3-fold StratifiedGroupKFold
        # rebuilt per outer fold from the outer training subset's groups.
        oof_prob = np.zeros(len(X_train), dtype=np.float64)
        for _tr_idx, _va_idx in inner_cv.split(X_train, y_train, groups=groups_train):
            _Xtr = X_train.iloc[_tr_idx]
            _ytr = y_train[_tr_idx]
            _Xva = X_train.iloc[_va_idx]
            _gtr = groups_train.iloc[_tr_idx]
            _cal_cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE).split(_Xtr, _ytr, groups=_gtr))
            _cal = CalibratedClassifierCV(clone(search.best_estimator_), cv=_cal_cv)
            _cal.fit(_Xtr, _ytr)
            oof_prob[_va_idx] = _cal.predict_proba(_Xva)[:, 1]
        oof_probs[name] = oof_prob

        # Final calibrator on the full X_train; Platt folds also spatially blocked.
        _final_cal_cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE).split(X_train, y_train, groups=groups_train))
        calibrated = CalibratedClassifierCV(search.best_estimator_, cv=_final_cal_cv)
        calibrated.fit(X_train, y_train)
        y_prob = calibrated.predict_proba(X_test)[:, 1]
        fitted_probs[name] = y_prob

        best_threshold, best_score, best_threshold_metrics = find_best_threshold(
            y_train, oof_prob, method="f1",
        )
        model_thresholds[name] = float(best_threshold)
        print(f"Tuned threshold for {name} on {grid} (OOF, train only): {best_threshold:.2f}")
        print(f"OOF metrics at chosen threshold: {best_threshold_metrics}")

        y_pred = (y_prob >= best_threshold).astype(int)

        metrics_dict = compute_metrics(y_test, y_pred, y_prob)
        cm = confusion_matrix(y_test, y_pred)

        print(f"{name} Best Params: {search.best_params_}")
        print(f"{name} Metrics: {metrics_dict}")
        print("Confusion Matrix:\n", cm)
        print(classification_report(y_test, y_pred, digits=4))

        save_confusion_matrix(
            cm,
            labels=["Non-Landslide (0)", "Landslide (1)"],
            title=f"{name} Confusion Matrix - {grid}",
            save_path=os.path.join(grid_out_dir, "plots", "cm", f"{name}_{grid}_cm.png")
        )

        save_roc_curve(
            y_test, y_prob,
            title=f"{name} ROC Curve - {grid}",
            save_path=os.path.join(grid_out_dir, "plots", "roc", f"{name}_{grid}_roc.png")
        )

        fitted_base = search.best_estimator_

        # BUG FIX: was X_train/y_train — permutation importance must use held-out test set
        importances = get_native_importance(name, fitted_base, X_test, y_test)
        plot_feature_importance(
            importances,
            X.columns,
            title=f"{name} Feature Importance - {grid}",
            save_path=os.path.join(grid_out_dir, "plots", "importance", f"{name}_{grid}_importance.png")
        )

        if name in ["RF", "XGB"]:
            plot_shap_tree(
                fitted_base,
                shap.sample(
                    X_test,
                    min(len(X_test), SHAP_EXPLAIN_SIZE),
                    random_state=RANDOM_STATE
                ),
                title=f"{name} SHAP - {grid}",
                save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"{name}_{grid}")
            )
        else:
            plot_shap_kernel(
                fitted_base,
                X_train,
                X_test,
                title=f"{name} SHAP - {grid}",
                save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"{name}_{grid}")
            )

        model_path = os.path.join(grid_out_dir, "models", f"{name}_{grid}_calibrated.pkl")
        joblib.dump(calibrated, model_path)

        params_path = os.path.join(grid_out_dir, "models", f"{name}_{grid}_best_params.json")
        save_json(search.best_params_, params_path)

        fitted_models[name] = calibrated

        # Spatial CV AUC (mean +/- std) on the training fold only - SAME protocol as the
        # grid/pixel notebook (StratifiedGroupKFold spatial blocks, scoring='roc_auc') so the
        # two mapping-unit systems are directly comparable.
        _cv_estimator = CalibratedClassifierCV(clone(search.best_estimator_), cv=3)
        mean_auc_cv, std_auc_cv, fold_aucs_cv = compute_spatial_cv_auc_stats(
            _cv_estimator, X_train, y_train, groups_train,
            n_splits=int(round(1.0 / TEST_SIZE)), random_state=RANDOM_STATE,
        )
        print(f"  {name} spatial CV AUC: {mean_auc_cv:.4f} +/- {std_auc_cv:.4f} "
              f"(folds {np.round(fold_aucs_cv, 4)})")

        grid_results.append({
            "grid": grid,
            "model": name,
            "threshold": float(best_threshold),
            **metrics_dict,
            "spatial_cv_auc_mean": mean_auc_cv,
            "spatial_cv_auc_std": std_auc_cv,
            "spatial_cv_auc_text": f"{mean_auc_cv:.4f} +/- {std_auc_cv:.4f}",
            "model_path": model_path,
        })

        gc.collect()

    # ======================================
    # 8. ENSEMBLE
    # ======================================
    print(f"\n--------------- Ensemble on {grid} ---------------")

    ensemble_prob = ensemble_predict_proba(fitted_models, X_test)
    _oof_ens = np.mean(
        np.column_stack([oof_probs[m] for m in ["RF", "SVM", "XGB"]]), axis=1
    )
    ens_threshold, _ens_score, _ens_metrics = find_best_threshold(
        y_train, _oof_ens, method="f1",
    )
    model_thresholds["Ensemble"] = float(ens_threshold)
    print(f"Ensemble OOF F1-optimal threshold: {ens_threshold:.2f}")
    print(f"OOF metrics at ensemble threshold: {_ens_metrics}")
    ensemble_pred = (ensemble_prob >= ens_threshold).astype(int)

    ensemble_metrics = compute_metrics(y_test, ensemble_pred, ensemble_prob)
    ensemble_cm = confusion_matrix(y_test, ensemble_pred)

    print("Ensemble Metrics:", ensemble_metrics)
    print("Ensemble Confusion Matrix:\n", ensemble_cm)
    print(classification_report(y_test, ensemble_pred, digits=4))

    save_confusion_matrix(
        ensemble_cm,
        labels=["Non-Landslide (0)", "Landslide (1)"],
        title=f"Ensemble Confusion Matrix - {grid}",
        save_path=os.path.join(grid_out_dir, "plots", "cm", f"Ensemble_{grid}_cm.png")
    )

    save_roc_curve(
        y_test, ensemble_prob,
        title=f"Ensemble ROC Curve - {grid}",
        save_path=os.path.join(grid_out_dir, "plots", "roc", f"Ensemble_{grid}_roc.png")
    )

    # Journal-quality combined-ROC panel (all four models overlaid)
    fitted_probs["Ensemble"] = ensemble_prob
    save_combined_roc(
        fitted_probs, y_test, grid_name=grid,
        save_path=os.path.join(grid_out_dir, "plots", "roc", f"Combined_{grid}_roc.png"),
    )

    plot_ensemble_permutation_importance(
        fitted_models,
        X_test,
        y_test,
        save_path=os.path.join(grid_out_dir, "plots", "importance", f"Ensemble_{grid}_importance.png"),
        title=f"Ensemble Permutation Importance - {grid}",
        threshold=ens_threshold,
    )

    plot_ensemble_shap(
        fitted_models, X_train, X_test,
        save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"Ensemble_{grid}"),
        title=f"Ensemble SHAP - {grid}",
    )

    ensemble_bundle = {
        "models": fitted_models,
        "features": list(X.columns),
        "threshold": float(ens_threshold),
        "per_model_thresholds": dict(model_thresholds),
        "method": "mean_probability",
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "spatial_block_size_m": SLOPE_UNIT_BLOCK_M,
        "scale_pos_weight_xgb": float(scale_pos_weight),
        "outer_split_strategy": "first fold of StratifiedGroupKFold(5) with spatial blocks",
    }
    ensemble_path = os.path.join(grid_out_dir, "models", f"Ensemble_{grid}.pkl")
    joblib.dump(ensemble_bundle, ensemble_path)

    grid_results.append({
        "grid": grid,
        "model": "Ensemble",
        "threshold": float(ens_threshold),
        **ensemble_metrics,
        "model_path": ensemble_path,
    })

    grid_summary_df = pd.DataFrame(grid_results)
    grid_summary_df.to_csv(os.path.join(grid_out_dir, f"{grid}_model_summary.csv"), index=False)

    all_grid_summary.extend(grid_results)

    print(f"\nCompleted grid: {grid}")
    print(grid_summary_df)

    gc.collect()

# ==========================================
# 9. FINAL SUMMARY ACROSS ALL GRIDS
# ==========================================
final_summary_df = pd.DataFrame(all_grid_summary)
final_summary_path = os.path.join(base_output_dir, "all_grids_model_summary.csv")
final_summary_df.to_csv(final_summary_path, index=False)

print("\nFinal summary across all grids:")
print(final_summary_df)
print(f"\nSaved final summary to: {final_summary_path}")
